In [1]:
%load_ext autoreload
%autoreload 2

```{toctree}
:maxdepth: 1
:caption: Contents:

# Walkthrough Example

First, different python libraries are imported to handle the data retrieval and processing.

In [ ]:
from datetime import datetime
import xarray as xr

The different modules in the `oceanicospy.downloads` subpackage are imported to showcase the data retrieval capabilities.

In [3]:
from oceanicospy.downloads import UHSLCDownloader,CMDSDownloader,ERA5Downloader

# University of Hawaii Sea Level Center (UHSLC) Data Downloader

This code is intended to download hourly sea-level data from the University of Hawaii Sea Level Center (UHSLC). The `UHSLCDownloader` class is defined to handle the downloading process. Three parameters are required: the `station_id` for the station of interest and the `output_path` where the downloaded data will be saved, and the `output_filename` for the resulting CSV file. The station id can be found on the UHSLC website (https://uhslc.soest.hawaii.edu/stations/).

A ``utc_offset_hours`` parameter is also included to convert the raw UHSLC timestamps to local time in the cleaned DataFrame. It follows the convention ``local - UTC``, so UTC-5 should be passed as ``-5`` (the default).

An start and end date can also be provided to trim the data after downloading. The dates should be passed as ``datetime`` objects (e.g. ``datetime(2010, 1, 1)``). If either of the dates is ``None``, the series will not be trimmed at that end.


In [4]:
UHSLCDownloader_obj = UHSLCDownloader(station_id='007', output_path='../data/temp/UHSLC', output_filename='station_007.csv')

In [5]:
filepath = UHSLCDownloader_obj.download()

Downloaded station_007.csv to ../data/temp/UHSLC/station_007.csv.


The method ``UHSLCDownloader.download()`` returns ``filepath``, which is the path to the downloaded file. The file is saved in CSV format and can be read using pandas or any other data analysis tool. A short method ``UHSLCDownloader.clean_data()`` is also included to quickly read the last downloaded file into a pandas DataFrame.

In [6]:
df_clean_local = UHSLCDownloader_obj.clean_data()

In [7]:
df_clean_local.head()

,depth[m]
1969-05-18 10:00:00,1.563
1969-05-18 11:00:00,1.399
1969-05-18 12:00:00,1.292
1969-05-18 13:00:00,1.277
1969-05-18 14:00:00,1.417


# Copernicus Marine Data Store (CMDS) downloader

There is a class called `CMDSDownloader` that is designed to download data from the Copernicus Marine Data Store (CMDS). This class uses the `copernicusmarine` toolbox to submit spatial/temporal subset requests to the CMDS and download the resulting data. The class is initialized with the following parameters:
- `dataset_id`: the ID of the dataset to download (e.g., `"cmems_mod_glo_wav_anfc_0.083deg_PT3H-i"`).
- `variables`: a list of variables to download (e.g., [``"variable1"``, ``"variable2"``]).
- spatial parameters for the region of interest: `lon_min`, `lon_max`, `lat_min`, `lat_max` (in degrees, EPSG:4326).
- temporal parameters for the time range of interest: `start_datetime_local` and `end_datetime_local`, passed as `datetime` objects.
- `utc_offset_hours`: local-time offset from UTC in hours, following the convention `local - UTC` (e.g., `-5` for UTC-5).
- `output_path`: the directory where the downloaded data will be saved.
- `output_filename` *(optional)*: the name of the output file (e.g., `"output.nc"`).
- `file_format` *(optional, default `"netcdf"`)*: the format of the output file (`"netcdf"` or `"zarr"`).


## For winds
There is a classmethod that tailors the CMDSDownloader to download wind data from the CMDS. This method is called ``CMDSDownloader.for_winds`` and it takes the same parameters as the main class, but it automatically sets the `dataset_id` to the appropriate dataset for wind data and selects the relevant variables for wind analysis. This method simplifies the process of downloading wind data from the CMDS by pre-configuring the necessary parameters for this specific type of data.

```{note}

For wind the following product and variables are defined as default:    
- Product: ``cmems_obs-wind_glo_phy_nrt_l4_0.125deg_PT1H``
- Variables: [``"eastward_wind"``, ``"northward_wind"``]

In [8]:
winds_CMDS = CMDSDownloader.for_winds(
    lon_min=-82,
    lon_max=-81,
    lat_min=12,
    lat_max=13,
    start_datetime_local=datetime(2023, 12, 1, 0, 0),             
    end_datetime_local=datetime(2023, 12, 31, 23, 0),               
    utc_offset_hours=-5,                                       
    output_path="../data/temp/CMDS",
    output_filename="winds_CMDS.nc",                             
    file_format="netcdf",
)

In [9]:
winds_CMDS_filepath = winds_CMDS.download()

INFO - 2026-04-20T02:26:11Z - Selected dataset version: "202207"
INFO - 2026-04-20T02:26:11Z - Selected dataset part: "default"


Downloaded winds_CMDS.nc to ../data/temp/CMDS/winds_CMDS.nc.


The user can verify the downloaded file contains the expected time in UTC.

In [11]:
winds_CMDS_ds_UTC = xr.open_dataset(winds_CMDS_filepath) 
winds_CMDS_ds_UTC

<xarray.Dataset> Size: 768kB
Dimensions:         (time: 744, latitude: 8, longitude: 8)
Coordinates:
  * latitude        (latitude) float32 32B 12.06 12.19 12.31 ... 12.81 12.94
  * longitude       (longitude) float32 32B -81.94 -81.81 ... -81.19 -81.06
  * time            (time) datetime64[ns] 6kB 2023-12-01T05:00:00 ... 2024-01...
Data variables:
    eastward_wind   (time, latitude, longitude) float64 381kB ...
    northward_wind  (time, latitude, longitude) float64 381kB ...
Attributes: (12/27)
    Conventions:                CF-1.6, ACDD-1.3
    date_created:               2022-06-23T01:30:26
    date_modified:              2022-06-23T01:30:26
    geospatial_lat_max:         89.9375
    geospatial_lat_min:         -89.9375
    geospatial_lat_resolution:  0.125
    ...                         ...
    references:                 Copernicus Marine Service Product User Manual...
    summary:                    Global ocean 10-m stress-equivalent wind and ...
    time_coverage_end:          2020-07-01T00:00:00
    time_coverage_start:        2020-07-01T00:00:00
    title:                       Global Ocean - Wind and Stress - Hourly - Fr...
    copernicusmarine_version:   2.3.0

With the method ``format_to_localtime``, the timestamps in the downloaded CMDS dataset can be converted from UTC to local time. This is done by applying the specified `utc_offset_hours` (e.g., `-5` for UTC-5) to the timestamps in the dataset. This method allows users to work with the data in their local time zone, which can be more convenient for analysis and interpretation.

In [12]:
winds_CMDS.format_to_localtime()

In [13]:
winds_CMDS_ds_localtime = xr.open_dataset(winds_CMDS_filepath) 
winds_CMDS_ds_localtime

<xarray.Dataset> Size: 768kB
Dimensions:         (time: 744, latitude: 8, longitude: 8)
Coordinates:
  * latitude        (latitude) float32 32B 12.06 12.19 12.31 ... 12.81 12.94
  * longitude       (longitude) float32 32B -81.94 -81.81 ... -81.19 -81.06
  * time            (time) datetime64[ns] 6kB 2023-12-01 ... 2023-12-31T23:00:00
Data variables:
    eastward_wind   (time, latitude, longitude) float64 381kB ...
    northward_wind  (time, latitude, longitude) float64 381kB ...
Attributes: (12/27)
    Conventions:                CF-1.6, ACDD-1.3
    date_created:               2022-06-23T01:30:26
    date_modified:              2022-06-23T01:30:26
    geospatial_lat_max:         89.9375
    geospatial_lat_min:         -89.9375
    geospatial_lat_resolution:  0.125
    ...                         ...
    references:                 Copernicus Marine Service Product User Manual...
    summary:                    Global ocean 10-m stress-equivalent wind and ...
    time_coverage_end:          2020-07-01T00:00:00
    time_coverage_start:        2020-07-01T00:00:00
    title:                       Global Ocean - Wind and Stress - Hourly - Fr...
    copernicusmarine_version:   2.3.0

## For waves

Similar to winds, there is a default configuration for downloading wave data from the CMDS. The classmethod ``CMDSDownloader.for_waves`` is designed to facilitate the download of wave data by pre-setting the appropriate `dataset_id` and selecting the relevant variables for wave analysis. This method allows users to easily access wave data from the CMDS without needing to manually configure the parameters.

```{note}

For waves the following product and variables are defined as default:    
- Product: ``cmems_mod_glo_wav_anfc_0.083deg_PT3H-i``
- Variables: [``"VHM0"``, ``"VMDR"``, ``"VTPK"``]

In [14]:
waves_CMDS = CMDSDownloader.for_waves(
    lon_min=-82,
    lon_max=-80,
    lat_min=12,
    lat_max=14,
    start_datetime_local=datetime(2024, 12, 1, 0, 0),             
    end_datetime_local=datetime(2024, 12, 31, 23, 0),               
    utc_offset_hours=-5,                                       
    output_path="../data/temp/CMDS",
    output_filename="waves_CMDS.nc",                             
    file_format="netcdf",
)

In [15]:
waves_CMDS_filepath = waves_CMDS.download()

INFO - 2026-04-20T02:26:27Z - Selected dataset version: "202411"
INFO:copernicusmarine:Selected dataset version: "202411"
INFO - 2026-04-20T02:26:27Z - Selected dataset part: "default"
INFO:copernicusmarine:Selected dataset part: "default"


Downloaded waves_CMDS.nc to ../data/temp/CMDS/waves_CMDS.nc.


Similar to the wind data, the user can verify the downloaded wave data contains the expected time in UTC. The method ``format_to_localtime`` can also be applied to wave data to convert the timestamps from UTC to local time using the specified `utc_offset_hours`.

In [16]:
waves_CMDS.format_to_localtime()

In [17]:
waves_CMDS_ds_localtime = xr.open_dataset(waves_CMDS_filepath) 
waves_CMDS_ds_localtime

<xarray.Dataset> Size: 4MB
Dimensions:    (time: 248, latitude: 25, longitude: 25)
Coordinates:
  * latitude   (latitude) float64 200B 12.0 12.08 12.17 ... 13.83 13.92 14.0
  * longitude  (longitude) float64 200B -82.0 -81.92 -81.83 ... -80.08 -80.0
  * time       (time) datetime64[ns] 2kB 2024-12-01T01:00:00 ... 2024-12-31T2...
Data variables:
    VHM0       (time, latitude, longitude) float64 1MB ...
    VMDR       (time, latitude, longitude) float64 1MB ...
    VTPK       (time, latitude, longitude) float64 1MB ...
Attributes: (12/17)
    Conventions:               CF-1.6
    area:                      GLO
    contact:                   servicedesk.cmems@mercator-ocean.eu
    credit:                    E.U. Copernicus Marine Service Information (CM...
    geospatial_lat_max:        90.0
    geospatial_lat_min:        -80.0
    ...                        ...
    geospatial_lon_units:      degree
    institution:               METEO-FRANCE
    producer:                  CMEMS - Global Monitoring and Forecasting Centre
    product:                   GLOBAL_ANALYSIS_FORECAST_WAV_001_027
    references:                http://marine.copernicus.eu
    copernicusmarine_version:  2.3.0

# ERA5 Downloader

ERA5 is the fifth generation ECMWF reanalysis for the global climate and weather. This product is provided by the European Centre for Medium-Range Weather Forecasts (ECMWF) and is available through the Copernicus Climate Data Store (CDS). The `ERA5Downloader` class is designed to facilitate the download of ERA5 data from the CDS. This class uses the `cdsapi` library to submit requests to the CDS and download the resulting data.

In [18]:
variables = ["10m_u_component_of_wind","10m_v_component_of_wind",]

# Spatial domain (WGS84); San Andrés example
lon_min, lon_max = -81.90, -81.50   # West/East
lat_min, lat_max =  12.35,  12.75   # South/North

start_local = datetime(2023, 8, 1, 0, 0)
end_local   = datetime(2023, 8, 31, 23, 0)
utc_offset_hours = -5  # Local = UTC-5

# Output file
output_path = "../data/temp/ERA5/"

winds_ERA5 = ERA5Downloader(
    variables=variables,
    lon_min=lon_min,
    lon_max=lon_max,
    lat_min=lat_min,
    lat_max=lat_max,
    start_datetime_local=start_local,
    end_datetime_local=end_local,
    utc_offset_hours=utc_offset_hours,
    output_path=output_path,
    output_filename="winds_ERA5.nc"
)

In [19]:
winds_ERA5_filepath = winds_ERA5.download()

2026-04-20 02:26:47,208 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
INFO:ecmwf.datastores.legacy_client:[2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-

7c10284c8f5aeda2736fa37fb04cc635.nc:   0%|          | 0.00/1.69M [00:00<?, ?B/s]

Downloaded winds_ERA5.nc to ../data/temp/ERA5/winds_ERA5.nc.


Similar to the CMDSDownloader, the method ``format_to_localtime`` is included to convert the timestamps in the downloaded ERA5 dataset from UTC to local time. This allows users to work with the data in their local time zone, which can be more convenient for analysis and interpretation.

In [20]:
winds_ERA5_ds_UTC = xr.open_dataset(winds_ERA5_filepath) 
winds_ERA5_ds_UTC

<xarray.Dataset> Size: 3MB
Dimensions:     (valid_time: 1464, latitude: 17, longitude: 17)
Coordinates:
    number      int64 8B ...
  * valid_time  (valid_time) datetime64[ns] 12kB 2023-08-01 ... 2023-09-30T23...
  * latitude    (latitude) float64 136B 12.75 12.72 12.7 ... 12.4 12.37 12.35
  * longitude   (longitude) float64 136B -81.9 -81.88 -81.85 ... -81.52 -81.5
    expver      (valid_time) <U4 23kB ...
Data variables:
    u10         (valid_time, latitude, longitude) float32 2MB ...
    v10         (valid_time, latitude, longitude) float32 2MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-04-17T23:43 GRIB to CDM+CF via cfgrib-0.9.1...

In [21]:
winds_ERA5.format_to_localtime()

The dataset is open with xarray and the dates should be in local time.

In [22]:
winds_ERA5_ds_localtime = xr.open_dataset(winds_ERA5_filepath)
winds_ERA5_ds_localtime

<xarray.Dataset> Size: 2MB
Dimensions:     (valid_time: 744, latitude: 17, longitude: 17)
Coordinates:
    number      int64 8B ...
  * valid_time  (valid_time) datetime64[ns] 6kB 2023-08-01 ... 2023-08-31T23:...
  * latitude    (latitude) float64 136B 12.75 12.72 12.7 ... 12.4 12.37 12.35
  * longitude   (longitude) float64 136B -81.9 -81.88 -81.85 ... -81.52 -81.5
    expver      (valid_time) <U4 12kB ...
Data variables:
    u10         (valid_time, latitude, longitude) float32 860kB ...
    v10         (valid_time, latitude, longitude) float32 860kB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-04-17T23:43 GRIB to CDM+CF via cfgrib-0.9.1...